# Clustering concepts by manifold shape

24 concepts x 10 prompt-template variants = 240 manifolds (gemma-2-2b, layer 6).
Each concept's signature is the mean of its 10 variants; concepts are then clustered
by that signature. Expected-topology labels are loaded for annotation only, never as
a clustering input.

Measurement lives in `shape_features.py`, figures in `figures.py`.

In [ ]:
import os, glob, json, io, contextlib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

from shape_features import shape_features

REPO = os.path.abspath('../..')
VARIANTS = os.path.join(REPO, 'outputs/activations/gemma-2-2b/variants')
SETS = json.load(open(os.path.join(REPO, 'concept_sets.json')))['sets']
CACHE, SEED, K = 'shape_features.csv', 0, 6

# Ten diagnostics, each aimed at a specific distinction: id_proj (line ~1 vs plane
# ~2), order/loop (does a cycle exist AND traverse in the concept's own value
# order), h1_gap (one dominant bar vs many), mst_* (path vs branching vs lattice),
# curv_* (clustered vs branching).
#
# order/loop replace the older b1_sig/b2_sig, which required a per-manifold null.
# The order test is calibrated once per n instead of once per manifold, so it is
# ~5x cheaper, and it is the stricter gate: it rejects `political` (a plane with a
# spurious loop that cleared its null) and recovers `days` (a real circle the null
# threshold missed).
FEATURES = ['id_proj', 'order', 'loop', 'h1_gap', 'mst_max_deg',
            'mst_frac_leaf', 'mst_frac_deg2',
            'curv_mean', 'curv_std', 'curv_frac_neg']

In [ ]:
def build():
    """One row per (concept, template variant). ~2 min for 240."""
    rows = []
    for f in sorted(glob.glob(os.path.join(VARIANTS, '*.npz'))):
        stem = os.path.basename(f)[:-4]
        concept, variant = stem.split('__')[0], stem.split('__')[1].rsplit('_layer', 1)[0]
        act = np.load(f, allow_pickle=True)['activations'].astype(np.float64)
        with contextlib.redirect_stdout(io.StringIO()):
            r = shape_features(act, seed=SEED)
        rows.append({'concept': concept, 'variant': variant,
                     'family': SETS[concept]['family'], **r})
    return pd.DataFrame(rows)


if not os.path.exists(CACHE):
    build().to_csv(CACHE, index=False)
df = pd.read_csv(CACHE)
print(f'{len(df)} manifolds, {df.concept.nunique()} concepts')
df.head()

In [ ]:
sig = df.groupby('concept')[FEATURES].mean()
fam = df.groupby('concept')['family'].first()

X = StandardScaler().fit_transform(np.asarray(sig.values, float))
groups = pd.Series(fcluster(linkage(pdist(X), 'average'), K, 'maxclust'), index=sig.index)

for g in sorted(groups.unique()):
    members = groups.index[groups == g]
    print(f'group {g}: ' + ', '.join(f'{m} [{fam[m]}]' for m in members))

In [ ]:
from figures import fig_point_cloud, fig_loop_significance

fig_point_cloud(df)         # what the clustering is looking at
fig_loop_significance(df)   # which concepts beat their own null